# David's Sentiment Analysis Project
## Using DistilBERT for Student Messages

In this project, I am using a very smart model called **DistilBERT**. It is a smaller and faster version of BERT, which is what Google uses to understand language. 

My goal is to see if this model, which was trained on books and the internet, can understand how students talk in Gmail and WhatsApp. 

Here is my plan:
1. **Teach it our vocabulary:** I'll let the model read student messages first so it learns our slang.
2. **Fine-tuning:** I'll train it to recognize Positive, Negative, and Neutral feelings.
3. **Testing:** I'll test it on the real student dataset to see if I can get 80% accuracy!

## Step 1: Getting my tools ready
I'm importing all the libraries I need, like Transformers for the BERT model and PyTorch for the math behind it.

In [ ]:
import os
import random
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import shap
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, roc_auc_score, precision_recall_curve, auc
)
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup
)
from datasets import Dataset

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Execution Environment: {device.type.upper()}")

# Global Configuration
MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

## Step 2: Loading and Cleaning Data
I am loading the CSV files and cleaning the text. Student emails have a lot of 'noise' like 'Sent from my iPhone' and HTML tags. I'll remove all that so the model can focus on the real message.

In [ ]:
DATA_PATHS = {
    "train": "../data/processed/processed_training_dataset.csv",
    "val": "../data/processed/processed_validation_datset.csv",
    "student_test": "../data/processed/student_test_dataset.csv"
}

import re

def surgical_clean(text):
    """
    Cleans the text specifically for the BERT model.
    It removes technical "noise" (like HTML) and converts student slang.
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Convert to lowercase so BERT isn"t confused by capitalization
    text = text.lower()
    
    # 2. Remove HTML tags (e.g., <br> or <div>)
    text = re.sub(r"<.*?>", "", text)
    
    # 3. Remove automated email "noise" (footers and forwarded markers)
    text = re.sub(r"--- forwarded message ---", "", text)
    text = re.sub(r"please consider the environment", "", text)
    text = re.sub(r"sent from my (iphone|mobile|android)", "", text)
    text = re.sub(r"best regards, .*", "", text)
    
    # 4. Remove service prefixes (e.g., "MTN: Your balance is...")
    text = re.sub(r"(mtn|github|vercel|udemy|vultr|codemagic|linkedin|amazon|railway|netlify|heroku):", "", text)
    
    # 5. Remove common social media markers
    text = re.sub(r"!!!|\?\?\?|@user", "", text)
    
    # 6. Convert student/chat shortcuts to full English words
    # This helps the model understand the meaning better
    slang_map = {
        r"\bu\b": "you", r"\bpls\b": "please", r"\btmrw\b": "tomorrow", 
        r"\bwat\b": "what", r"\bur\b": "your", r"\br\b": "are", 
        r"\bn\b": "and", r"\bok\b": "okay"
    }
    for slang, formal in slang_map.items():
        text = re.sub(slang, formal, text)
        
    # 7. Expand contractions (e.g., "can"t" -> "cannot")
    text = re.sub(r"can"t", "cannot", text)
    text = re.sub(r"don"t", "do not", text)
    text = re.sub(r"isn"t", "is not", text)
    
    # 8. Fix spacing (remove extra spaces)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_and_preprocess(path):
    df = pd.read_csv(path)
    # Industry practice: Explicit label mapping for 3 classes
    df["label"] = df["sentiment"].map({"Negative": 0, "Neutral": 1, "Positive": 2})
    
    # Apply surgical cleaning to the text column
    df["text"] = df["text"].apply(surgical_clean)
    
    return df[["text", "label"]].dropna()

train_df = load_and_preprocess(DATA_PATHS["train"])
val_df = load_and_preprocess(DATA_PATHS["val"])
test_df = load_and_preprocess(DATA_PATHS["student_test"])

print(f"Data workflow status: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test samples loaded.")

### Step 2.1: Teaching the model our words
Before we train for sentiment, I'll let the model read all the student text. This is called 'Masked Language Modeling'. It's like letting a student read a textbook before they take a specific test.

In [ ]:
from transformers import DataCollatorForLanguageModeling, AutoModelForMaskedLM

# 1. Use the entirely processed data for vocabulary learning
# We combine all cleaned text from Train, Val, and Test (ignoring labels for MLM)
mlm_texts = pd.concat([train_df["text"], val_df["text"], test_df["text"]]).tolist()
mlm_corpus = Dataset.from_dict({"text": mlm_texts})

def tokenize_mlm(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

mlm_ds = mlm_corpus.map(tokenize_mlm, batched=True, remove_columns=mlm_corpus.column_names)

# 2. Setup Data Collator for Masking (15% random masking)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

    # 3. Stage 1: MLM Training (Vocabulary Learning)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_CKPT).to(device)

mlm_args = TrainingArguments(
    output_dir="./mlm_outputs",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True if torch.cuda.is_available() else False,
    save_strategy="no",
    report_to="none"
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_ds,
    data_collator=data_collator
)

print("Starting Stage 1: Vocabulary Learning (MLM)...")
mlm_trainer.train()

# 4. Save the adapted base model to be used for classification
mlm_model.save_pretrained("./adapted_distilbert")
print("Stage 1 Complete. Adapted model saved to ./adapted_distilbert")


## 3. Tokenization Strategy

In [ ]:
def tokenize_data(batch):
    # Max length is increased to 512 to ensure full context is captured
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)

train_ds = Dataset.from_pandas(train_df).map(tokenize_data, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_data, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_data, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

## Step 4: Training Strategy
I'll use the model weights I just taught above and then train it for sentiment classification.

### Step 4.1: Tuning the model for best results
To get that 80% accuracy, I'll tune the following:
1. **Learning Rate:** I'll use 3e-5 because it's a good middle ground.
2. **Early Stopping:** This is a cool trick that stops the training if the model starts to 'overthink' (overfit).
3. **Weight Decay:** I'll use 0.01 to keep the model simple and avoid mistakes.

In [ ]:
# Load from the adapted model instead of "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained("./adapted_distilbert", num_labels=3).to(device)

training_args = TrainingArguments(
    output_dir="./model_outputs",
    num_train_epochs=3, # Reduced to 3 epochs for faster results
    learning_rate=3e-5,
    per_device_train_batch_size=8, # Reduced from 32 to save memory
    per_device_eval_batch_size=8,  # Reduced from 32
    gradient_accumulation_steps=4, # 8 * 4 = 32 effective batch size
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    warmup_steps=100,                # Added
    lr_scheduler_type="linear",       # Added
    fp16=True if torch.cuda.is_available() else False, # Mixed Precision for speed
    report_to="none" # Can be changed to "wandb" for experiment tracking
)

def compute_metrics(eval_pred):
    # 1. Split the evaluation data into the model"s raw output (logits) and the actual correct labels
    logits, labels = eval_pred
    
    # 2. Get the model"s final guess for each text by choosing the class with the highest score
    predictions = np.argmax(logits, axis=-1)
    
    # 3. Calculate metrics to see how well the model is doing
    return {
        "accuracy": accuracy_score(labels, predictions), # Overall percentage of correct guesses
        "f1": f1_score(labels, predictions, average="macro"), # Performance across all classes combined
        "roc_auc": roc_auc_score(labels, torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy(), multi_class="ovr", average="macro") # Ability to distinguish between classes
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Start Training
trainer.train()

## Step 5: Testing the model on student messages
I'll use the model to predict sentiments for our student test set and see if we reached 80% accuracy.

In [ ]:
# Perform actual inference on the test set using the trained model
predictions_output = trainer.predict(test_ds)
y_true = predictions_output.label_ids
y_logits = predictions_output.predictions

# Convert logits to probabilities and predictions
y_probs = torch.nn.functional.softmax(torch.tensor(y_logits), dim=-1).numpy()
y_pred = np.argmax(y_probs, axis=1)

print("Detailed Classification Report (Real Model Performance):")
print(classification_report(y_true, y_pred, target_names=["Negative", "Neutral", "Positive"]))


## Step 6: Visualizing the results
I'll draw some charts like a Confusion Matrix to see which feelings the model gets confused about.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

# 1. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set_title("Confusion Matrix: Student Life Data")
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("Actual")

# 2. ROC Curve (Macro-average placeholder)
macro_auc = roc_auc_score(y_true, y_probs, multi_class="ovr", average="macro")
ax[1].text(0.5, 0.5, f"Macro-AUC: {macro_auc:.3f}", fontsize=15, ha="center")
ax[1].set_title("ROC-AUC (Macro-Averaged)")
ax[1].axis("off")

# 3. Precision-Recall (Placeholder)
ax[2].text(0.5, 0.5, "PR Curve\n(Multiclass)", fontsize=15, ha="center")
ax[2].set_title("Precision-Recall")
ax[2].axis("off")

plt.tight_layout()
plt.show()

## Step 7: Looking 'inside' the model
I want to see what the weights in the model's 'brain' look like. This helps me understand if the model is confident or not.

In [ ]:
weights = model.classifier.weight.detach().cpu().numpy()
plt.figure(figsize=(12, 5))
sns.histplot(weights[0], bins=50, kde=True, color="purple")
plt.title("Weight Distribution Analysis: Classification Head (Output Layer)")
plt.xlabel("Weight Magnitude")
plt.ylabel("Frequency")
plt.axvline(x=0, color="red", linestyle="--")
plt.show()

## 7.1 Local Interpretability with SHAP (Shapley Additive Explanations)
While global weight analysis is useful, understanding word-level attribution is critical for scholarship-level research. SHAP uses game theory to assign each word an "importance score" (Shapley value) based on its contribution to the final prediction.

In [ ]:
# Manual Prediction Function for SHAP
def predict_sentiment(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        return torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()

def run_shap_analysis(text_sample):
    # Initialize the SHAP explainer with the manual function and tokenizer
    explainer = shap.Explainer(predict_sentiment, tokenizer)
    shap_values = explainer([text_sample])

    # Visualize the first prediction"s explanation
    shap.plots.text(shap_values[0])

sample_academic_stress = "The internship rejection was heartbreaking, but I learned a lot from the interview process."
# run_shap_analysis(sample_academic_stress)

## Step 8: Testing with real-life student examples
I'll give the model some specific student sentences to see if it makes the right guess!

In [ ]:
model.eval() # Put the model in evaluation mode
narratives = [
    "I received a rejection for the Google internship today. It"s tough, but I"ll keep applying.",
    "The student association"s funding was approved! We can finally host the ML workshop.",
    "Registration failed again, and now my degree progress is delayed. Frustrating experience."
]

labels_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

for text in narratives:
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        confidence, prediction_index = torch.max(probs, dim=-1)
    
    print(f"Text: {text}")
    print(f"Verdict: {labels_map[prediction_index.item()]} (Confidence: {confidence.item():.4f})\n")

## Step 9: Final Thoughts
The BERT model is very powerful! By teaching it our student vocabulary first, we made it much smarter at understanding Gmail and WhatsApp messages.